<a href="https://colab.research.google.com/github/fabio-la-torre/smartphone-trade-in-price-estimator/blob/main/trade_in_price_estimator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [125]:
import pandas as pd
import numpy as np

# Load the dataset directly from this GitHub repository
url = "https://raw.githubusercontent.com/fabio-la-torre/smartphone-trade-in-price-estimator/main/used_device_data.csv"
df = pd.read_csv("used_device_data.csv")
total = len(df)

df = df[df["screen_size"] <= 17.8]  # keep only phones (screen up to 7 inches)

print("Devices in dataset:", total)
print("Tablets removed:", total - len(df))
print("Phones left:", len(df))
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'used_device_data.csv'

In [ ]:
df.corr(numeric_only=True)["normalized_used_price"].sort_values()

In [ ]:
# Feature selection experiments (phones only)
# Each experiment trains a model with a different set of features and prints its R² score
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

df["5g"] = df["5g"] == "yes"   # convert yes/no to True/False
df["4g"] = df["4g"] == "yes"

def test_features(feature_list):
    d = df[feature_list + ["normalized_used_price"]].dropna()
    X_tr, X_te, y_tr, y_te = train_test_split(d[feature_list], d["normalized_used_price"],
                                              test_size=0.2, random_state=42)
    m = LinearRegression()
    m.fit(X_tr, y_tr)
    return round(m.score(X_te, y_te), 3)

base = ["normalized_new_price", "release_year", "days_used", "ram"]

experiments = {
    "1. Base + battery + 5G":                 base + ["battery", "5g"],
    "2. Internal memory instead of battery":  base + ["internal_memory", "5g"],
    "3. Add cameras":                         base + ["battery", "5g", "rear_camera_mp", "front_camera_mp"],
    "4. 4G instead of 5G":                    base + ["battery", "4g", "rear_camera_mp", "front_camera_mp"],
    "5. No 4G/5G (simpler, same score)":      base + ["battery", "rear_camera_mp", "front_camera_mp"],
    "6. Add screen size (final model)":       base + ["battery", "screen_size", "rear_camera_mp", "front_camera_mp"],
}

for name, feature_list in experiments.items():
    print(test_features(feature_list), "-", name)

In [ ]:
# Train and evaluate the model
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# Features chosen after the experiments (see previous cell)
features = ["normalized_new_price", "release_year", "days_used", "ram",
            "battery", "screen_size", "rear_camera_mp", "front_camera_mp"]

# Keep only phones that have a value for every feature
data = df[features + ["normalized_used_price"]].dropna()
print("Phones with complete data:", len(data))

X = data[features]                   # inputs: phone characteristics
y = data["normalized_used_price"]    # target: used price (log scale)

# Split the phones: 80% to train the model, 20% to test it on phones it has never seen
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Training set:", len(X_train), "phones (80%)")
print("Test set:", len(X_test), "phones (20%)")

# Training: the model learns one weight for each feature
model = LinearRegression()
model.fit(X_train, y_train)

# R² score on the test set: share of price differences the model explains (1.0 = perfect)
score = model.score(X_test, y_test)
print("R² score on test set:", round(score, 3))

In [ ]:
# Weights learned by the model (effect of +1 unit of each feature on the log price)
for name, weight in zip(features, model.coef_):
    print(f"{name}: {weight:.6f}")

In [ ]:
# Estimate the trade-in value of a phone in euros
def estimate_price(new_price_eur, release_year, days_used, ram, battery_mah,
                   screen_inches, rear_mp, front_mp):
    phone = pd.DataFrame([{
        "normalized_new_price": np.log(new_price_eur),  # euros -> log scale used by the model
        "release_year": release_year,
        "days_used": days_used,
        "ram": ram,                                     # GB
        "battery": battery_mah,                         # mAh
        "screen_size": screen_inches * 2.54,            # inches -> cm
        "rear_camera_mp": rear_mp,
        "front_camera_mp": front_mp,
    }])
    log_price = model.predict(phone)[0]
    return round(np.exp(log_price))                     # log scale -> euros

# Example: 300 € phone, released 2020, used 300 days, 4 GB RAM, 4000 mAh, 6.2", 10 MP + 10 MP
print(estimate_price(300, 2020, 300, 4, 4000, 6.2, 10, 10), "€")